# PydanticAI Tutorial: Building AI Agents with Structure and Type Safety

**Target audience:** Python beginners with basic knowledge of functions and classes  
**Theme:** Astronomical object classification and analysis  
**Estimated time:** 45 minutes

---

## What is PydanticAI?

When we call a Large Language Model (LLM) like GPT or Claude, we typically get back plain text. This is fine for chatbots, but problematic when we want structured, reliable data — for example, a catalog entry with specific fields, validated units, and consistent formatting.

**PydanticAI** solves this by combining two ideas:

1. **Pydantic** — a Python library that defines data schemas and validates data against them.
2. **Agents** — autonomous units that interact with an LLM, optionally calling external tools, until they produce a validated final answer.

The result: LLM outputs that behave like well-typed Python objects, not raw strings.

---

## Table of Contents

1. [Installation and Setup](#1-installation-and-setup)
2. [Section 1 — Basic Agents & Structured Outputs](#section-1--basic-agents--structured-outputs)
3. [Section 2 — Tool / Function Calling](#section-2--tool--function-calling)
4. [Section 3 — Dependency Injection & Context](#section-3--dependency-injection--context)
5. [Section 4 — Multi-Agent Workflows](#section-4--multi-agent-workflows)
6. [Summary and Next Steps](#summary-and-next-steps)

---
## 1. Installation and Setup

Install PydanticAI and the Anthropic provider (you may substitute `openai` or `google-generativeai` as needed).

In [19]:
import sys
sys.version

'3.13.5 | packaged by Anaconda, Inc. | (main, Jun 12 2025, 16:09:02) [GCC 11.2.0]'

In [20]:
# Install PydanticAI with Anthropic support
# Run this cell once, then restart the kernel if needed.
#%pip install "pydantic-ai[anthropic]" --quiet

In [21]:
import os

# Set your API key. In a real project, load this from a .env file
# or an environment variable — never hard-code secrets.
os.environ["ANTHROPIC_API_KEY"] = open("claude_FF.key").read().strip()#"your-api-key-here"

# Uncomment and use this instead if you prefer OpenAI:
# os.environ["OPENAI_API_KEY"] = "your-api-key-here"

print("Environment configured.")

Environment configured.


> **Note on model strings:** PydanticAI uses the format `"provider:model-name"`, for example:
> - `"anthropic:claude-sonnet-4-6"` 
> - `"openai:gpt-4o"`
> - `"google-gla:gemini-1.5-pro"`
>
> You can switch providers by changing this single string — the rest of your code remains unchanged.

---
## Section 1 — Basic Agents & Structured Outputs

### 1.1 The Simplest Possible Agent

An `Agent` wraps an LLM, a set of instructions, and optionally a schema for its output. Let us start with a plain-text agent.

In [22]:
from pydantic_ai import Agent

# Create an agent. The first argument is the model string.
# `instructions` is the system prompt — it tells the LLM what role it plays.
simple_agent = Agent(
    "anthropic:claude-sonnet-4-6",
    instructions="You are a knowledgeable astronomy assistant. Be concise.",
)

# `agent.run(...)` sends a user message and returns a result object;
# the text output lives in `.output`.
result = await simple_agent.run("What is a pulsar?")

print(result.output)

A **pulsar** is a highly magnetized, rapidly rotating **neutron star** that emits beams of electromagnetic radiation from its magnetic poles. As it spins, the beam sweeps past Earth like a lighthouse, producing regular, precise pulses of radiation — hence the name. Pulse periods range from milliseconds to several seconds. They are among the most accurate natural clocks in the universe.


**What just happened?**

PydanticAI sent the instructions and user message to the LLM, received a text response, and returned it wrapped in an `AgentRunResult`. Because we did not specify an `output_type`, the output is a plain `str`.

### 1.2 Structured Output with Pydantic Models

Plain text is unpredictable. If we ask for a star's spectral type and luminosity class, the model might return `"G2V"` one time and `"Spectral type G2, luminosity class V"` the next. 

We solve this by defining a **Pydantic model** — a class that declares exactly what fields we expect and what types they should have.

Let's try a naive call.

In [23]:
# Reuse simple_agent from above — same instructions, more demanding prompt.
result = await simple_agent.run("""
You are an expert stellar classifier.
Return the following classification parameters for the star Betelgeuse:
Name or designation of the stellar object,
Spectral type, e.g. G2V or M5III,
Effective surface temperature in Kelvin,
Luminosity in units of solar luminosity (L_sun),
Is the star on the main sequence?
""")

print(result.output)

Here are the classification parameters for **Betelgeuse (α Orionis)**:

| Parameter | Value |
|---|---|
| **Name/Designation** | Betelgeuse / Alpha Orionis / HD 39801 |
| **Spectral Type** | M1–M2 Ia (semi-regular variable) |
| **Effective Temperature** | ~3,500 K |
| **Luminosity** | ~100,000 L☉ |
| **Main Sequence?** | **No** — it is a red supergiant (luminosity class Ia) |

**Notes:** Betelgeuse is a massive evolved star (~15–20 M☉) that has left the main sequence and expanded enormously (~700–1,000 R☉). It is a well-known semi-regular pulsating variable and a candidate for a future core-collapse supernova.


To extract the relevant values we would have to write our own parser, which would have to handle many exceptions. 

Instead, let pydanticAI do that part.

In [24]:
from pydantic import BaseModel, Field
from pydantic_ai import Agent
from typing import Optional


# --- Define the output schema ---
# BaseModel is the Pydantic base class. Every field has a type annotation.
# Field(...) lets us add a human-readable description, which is passed
# to the LLM so it knows what each field means.
class StellarClassification(BaseModel):
    object_name: str = Field(description="Name or designation of the stellar object")
    spectral_type: str = Field(description="Spectral type, e.g. G2V or M5III")
    effective_temperature_K: int = Field(description="Effective surface temperature in Kelvin")
    luminosity_solar: float = Field(description="Luminosity in units of solar luminosity (L_sun)")
    is_main_sequence: bool = Field(description="True if the star is on the main sequence")
    notable_feature: Optional[str] = Field(
        default=None,
        description="One interesting or unusual feature of the object, if any"
    )


# --- Create an agent that returns a StellarClassification ---
# Setting `output_type` tells PydanticAI to validate the LLM's response
# against this schema. If validation fails, the agent automatically
# asks the LLM to try again.
classifier_agent = Agent(
    "anthropic:claude-sonnet-4-6",
    output_type=StellarClassification,
    instructions=(
        "You are an expert stellar classifier. "
        "Given a star name, return its classification parameters."
    ),
)

result = await classifier_agent.run("Classify the star Betelgeuse.")

# `result.output` is now a StellarClassification instance, not a string!
star = result.output
print(f"Object       : {star.object_name}")
print(f"Spectral type: {star.spectral_type}")
print(f"Temperature  : {star.effective_temperature_K} K")
print(f"Luminosity   : {star.luminosity_solar:.0f} L_sun")
print(f"Main sequence: {star.is_main_sequence}")
print(f"Notable      : {star.notable_feature}")

Object       : Betelgeuse
Spectral type: M2Iab
Temperature  : 3500 K
Luminosity   : 126000 L_sun
Main sequence: False
Notable      : Betelgeuse is a red supergiant and one of the largest known stars by radius. It is a semi-regular variable star and a strong candidate for a future core-collapse supernova. It gained public attention in 2019–2020 due to a dramatic dimming event known as the "Great Dimming."


**The output is a proper Python object.** You can access its fields, pass it to a function, store it in a database, or serialize it to JSON — no string parsing required.

> *Exercise:* loop over `["Sirius", "Vega", "Proxima Centauri", "Rigel"]`, call `classifier_agent.run(...)` for each, and build a `pd.DataFrame([r.output.model_dump() for r in results])`. Every column will already be correctly typed.

## Section 2 — Tool / Function Calling

An agent that only relies on the LLM's training data has a knowledge cutoff and cannot access real data. 

**Tools** (also called function tools) solve this: they are Python functions that the LLM can decide to call during its reasoning process.

The typical use cases are:
- Looking up a value in a catalog or database
- Calling an external API
- Performing a computation the LLM should not do itself

### 2.1 Built-in tools

PydanticAI comes with many default tools that the agent may decide to use for a particular task:

* WebSearchTool: Allows agents to search the web
* CodeExecutionTool: Enables agents to execute code in a secure environment
* ImageGenerationTool: Enables agents to generate images
* WebFetchTool: Enables agents to fetch web pages
* MemoryTool: Enables agents to use memory
* MCPServerTool: Enables agents to use remote MCP servers with communication handled by the model provider
* FileSearchTool: Enables agents to search through uploaded files using vector search (RAG)

Let's try the WebSearchTool

In [25]:
from pydantic_ai import Agent, WebSearchTool

agent = Agent('anthropic:claude-sonnet-4-6', builtin_tools=[WebSearchTool()])

result = await agent.run('Give me a sentence with the biggest news in Chile this week.')
print(result.output)

The biggest news in Chile this week is that President José Antonio Kast's economic reform plan to revive the country's economy is drawing criticism over its potential fiscal impact, as the IMF simultaneously lowered its growth projections for Chile.


### 2.2 Registering Your Own Tools with `@agent.tool_plain`

You can also register your own Python functions as tools.

Use `@agent.tool_plain` when the tool does not need access to the agent's runtime context (we cover context in Section 3). Tools can do anything Python can do — query a catalog, hit an API, perform a deterministic computation. The LLM sees the function name, its docstring, and its argument types, and decides on its own when to call it.

The example below shows a multi-step tool flow: the agent looks up a galaxy in a local catalog, then computes a derived quantity from the result.

In [26]:
from pydantic_ai import Agent
from pydantic import BaseModel, Field
from typing import Optional

# Simulated catalog (in practice this would be a database query or astroquery call)
CATALOG = {
    "m31": {"common_name": "Andromeda Galaxy", "type": "Spiral Galaxy",
             "distance_kpc": 770.0, "redshift": -0.001},
    "ngc4889": {"common_name": "NGC 4889", "type": "Elliptical Galaxy",
                "distance_kpc": 103_000.0, "redshift": 0.0217},
    "m87": {"common_name": "Messier 87", "type": "Elliptical Galaxy",
             "distance_kpc": 16_400.0, "redshift": 0.00436},
    "m51": {"common_name": "Whirlpool Galaxy", "type": "Spiral Galaxy",
             "distance_kpc": 7_620.0, "redshift": 0.00154},
}


class GalaxyReport(BaseModel):
    catalog_id: str
    common_name: str
    galaxy_type: str
    distance_mpc: float = Field(description="Distance in megaparsecs")
    redshift: float
    recessional_velocity_km_s: float = Field(
        description="Recessional velocity in km/s, computed from redshift"
    )
    summary: str


galaxy_agent = Agent(
    "anthropic:claude-sonnet-4-6",
    output_type=GalaxyReport,
    instructions=(
        "You are an extragalactic astronomy assistant. "
        "Use the catalog tool to look up galaxy data, then compute derived quantities."
    ),
)


@galaxy_agent.tool_plain
def lookup_galaxy(catalog_id: str) -> dict:
    """Look up a galaxy in the local catalog by its ID.
    
    Args:
        catalog_id: The catalog identifier (e.g. 'm31', 'ngc4889'). 
                    Use lowercase with no spaces.
    """
    key = catalog_id.lower().replace(" ", "")
    if key not in CATALOG:
        return {"error": f"Object '{catalog_id}' not found in catalog."}
    return CATALOG[key]


@galaxy_agent.tool_plain
def redshift_to_velocity(redshift: float) -> float:
    """Compute non-relativistic recessional velocity in km/s from redshift z.
    
    Args:
        redshift: The spectroscopic redshift z (dimensionless).
    """
    C_KM_S = 299_792.458  # speed of light in km/s
    return redshift * C_KM_S


result = await galaxy_agent.run("Give me a report on M87.")
report = result.output

print(f"Object     : {report.common_name} ({report.catalog_id.upper()})")
print(f"Type       : {report.galaxy_type}")
print(f"Distance   : {report.distance_mpc:.1f} Mpc")
print(f"Redshift   : {report.redshift}")
print(f"Velocity   : {report.recessional_velocity_km_s:.1f} km/s")
print(f"Summary    : {report.summary}")

Object     : Messier 87 (M87)
Type       : Elliptical Galaxy
Distance   : 16.4 Mpc
Redshift   : 0.00436
Velocity   : 1307.1 km/s
Summary    : Messier 87 (M87) is a giant elliptical galaxy located in the Virgo Cluster, one of the most massive and well-studied galaxies in the local universe. Here are its key properties:

- **Catalog ID:** M87
- **Type:** Elliptical Galaxy
- **Distance:** 16.4 Mpc (~53 million light-years)
- **Redshift (z):** 0.00436
- **Recessional Velocity:** ~1,307 km/s

M87 is famous for several remarkable features:
1. **Supermassive Black Hole:** M87 hosts one of the most massive known black holes, estimated at ~6.5 billion solar masses. It was the first black hole ever to be directly imaged, by the Event Horizon Telescope (EHT) in 2019.
2. **Relativistic Jet:** M87 produces a prominent relativistic plasma jet extending thousands of light-years, powered by the central black hole.
3. **Virgo Cluster Dominance:** As the largest galaxy near the center of the Virgo Clust

**What happened internally?**

1. PydanticAI sent the user message plus the schemas for both tools to the LLM.
2. The LLM decided to call `lookup_galaxy("m87")` — PydanticAI ran the Python function and sent the dict back.
3. Reading the returned redshift, the LLM then called `redshift_to_velocity(0.00436)`.
4. Finally the LLM produced a `GalaxyReport`-shaped response, which PydanticAI validated.

The LLM never did the arithmetic itself — it delegated to your deterministic Python code. It inferred which tools to use and the right argument order from the docstrings.

---
## Section 3 — Dependency Injection & Context

Tools decorated with `@agent.tool_plain` cannot access any runtime state. But what if your tools need shared resources — a database connection, an API key, a user's session data, or a configuration object?

PydanticAI's **dependency injection** system solves this cleanly. You:

1. Define a **dependencies dataclass** containing the shared resources.
2. Pass it to `Agent(..., deps_type=MyDeps)`.
3. Use `@agent.tool` (not `tool_plain`) and declare `ctx: RunContext[MyDeps]` as the first argument.
4. Access the dependencies through `ctx.deps`.

At run time, pass a `MyDeps` instance to `await agent.run(..., deps=deps)`.

In [27]:
from dataclasses import dataclass
from pydantic import BaseModel, Field
from pydantic_ai import Agent, RunContext


# --- Step 1: Define dependencies ---
# Shared context available to every tool and every dynamic instruction
# during a single agent run. We pack a realistic instrument model in here.
@dataclass
class ObservatoryDeps:
    observer_name: str
    telescope_aperture_m: float        # primary mirror diameter (m)
    read_noise_e: float                # per-pixel read noise (electrons)
    pixel_scale_arcsec: float          # plate scale (arcsec / pixel)
    seeing_fwhm_arcsec: float          # typical seeing FWHM (arcsec)
    sky_mag_v_per_arcsec2: float       # V-band sky brightness (mag / arcsec²)
    system_throughput: float           # end-to-end efficiency (optics × QE × filter × atm)
    log: list                           # tools can append entries here


class ObservationPlan(BaseModel):
    target: str
    recommended_exposure_s: float
    limiting_magnitude: float = Field(description="Limiting V magnitude reached at SNR=5")
    notes: str


# --- Step 2: Create the agent, declaring the dependency type ---
planner_agent = Agent(
    "anthropic:claude-sonnet-4-6",
    deps_type=ObservatoryDeps,
    output_type=ObservationPlan,
    instructions=(
        "You are an observation planning assistant for optical telescopes. "
        "Use the tool to estimate the limiting magnitude for trial exposure "
        "times and converge on what the observer needs."
    ),
)


# --- Step 3: Dynamic instruction that reads the dependencies ---
# Runs before each agent turn — injects context-aware text into the system prompt.
@planner_agent.instructions
def telescope_context(ctx: RunContext[ObservatoryDeps]) -> str:
    d = ctx.deps
    return (
        f"Observer: {d.observer_name}. "
        f"Telescope: {d.telescope_aperture_m} m aperture. "
        f"Read noise {d.read_noise_e} e-/pix, pixel scale {d.pixel_scale_arcsec} arcsec/pix, "
        f"seeing {d.seeing_fwhm_arcsec} arcsec, sky {d.sky_mag_v_per_arcsec2} mag/arcsec² (V), "
        f"throughput {d.system_throughput:.0%}."
    )


# --- Step 4: A tool that reads from ctx.deps ---
@planner_agent.tool
def estimate_limiting_magnitude(
    ctx: RunContext[ObservatoryDeps],
    exposure_s: float,
    snr_threshold: float = 5.0,
) -> float:
    """Estimate the V-band limiting magnitude reached in `exposure_s` seconds
    using the CCD equation.

    Accounts for source shot noise, sky-background shot noise, and read noise
    integrated over a photometric aperture of radius 1.5 × seeing FWHM.

    Args:
        exposure_s: Exposure time in seconds.
        snr_threshold: Required SNR (default 5, i.e. 5σ detection).
    """
    import math
    d = ctx.deps

    # V-band photon flux above the atmosphere for an m=0 source (Vega zero point)
    F0_PHOT = 8.8e4  # photons / s / cm² for m = 0
    A_cm2 = math.pi * (d.telescope_aperture_m * 100.0 / 2.0) ** 2
    zp_rate = F0_PHOT * d.system_throughput * A_cm2          # e-/s for m=0

    # Sky background per pixel (e-/s/pix)
    B = zp_rate * d.pixel_scale_arcsec**2 * 10 ** (-0.4 * d.sky_mag_v_per_arcsec2)

    # Photometric aperture in pixels (radius = 1.5 × FWHM)
    r_arcsec = 1.5 * d.seeing_fwhm_arcsec
    n_pix = math.pi * (r_arcsec / d.pixel_scale_arcsec) ** 2

    # Solve the CCD equation SNR² = (S t)² / (S t + n_pix (B t + RN²)) for S
    SNR = snr_threshold
    t = exposure_s
    noise = n_pix * (B * t + d.read_noise_e**2)
    S = SNR * (SNR + math.sqrt(SNR**2 + 4 * noise)) / (2 * t)

    m_lim = -2.5 * math.log10(S / zp_rate)
    d.log.append(
        f"t={t:.0f}s → m_lim={m_lim:.2f}  "
        f"(n_pix={n_pix:.0f}, B={B:.3f} e/s/pix)"
    )
    return round(m_lim, 2)


# --- Step 5: Run the agent, passing a concrete deps instance ---
session_log = []
deps = ObservatoryDeps(
    observer_name="Edwin Hubble",
    telescope_aperture_m=1.0,        # 1-metre telescope
    read_noise_e=5.0,
    pixel_scale_arcsec=0.5,
    seeing_fwhm_arcsec=1.2,
    sky_mag_v_per_arcsec2=21.0,      # dark site
    system_throughput=0.25,          # 25% end-to-end (optics × QE × filter × atm)
    log=session_log,
)

result = await planner_agent.run(
    "I want to detect V = 21 point sources at 5σ in a single exposure. "
    "What exposure time should I use?",
    deps=deps,
)

plan = result.output
print(f"Target              : {plan.target}")
print(f"Exposure            : {plan.recommended_exposure_s:.0f} s")
print(f"Limiting magnitude  : {plan.limiting_magnitude}")
print(f"Notes               : {plan.notes}")
print()
print("Session log:")
for entry in session_log:
    print(f"  - {entry}")

Target              : V = 21 point source at 5σ
Exposure            : 580 s
Limiting magnitude  : 21.08
Notes               : Converged via bracketing: 300s reaches only V=20.62, while 600s already exceeds the target at V=21.11. Fine-tuning shows ~580s yields a 5σ limiting magnitude of V=21.08, comfortably meeting the V=21 requirement with a small margin. Telescope assumed: 1.0 m aperture, 25% throughput, 5 e-/pix read noise, 0.5 arcsec/pix scale, 1.2 arcsec seeing, sky = 21.0 mag/arcsec² (V-band). Photometric aperture radius = 1.5 × FWHM.

Session log:
  - t=300s → m_lim=20.62  (n_pix=41, B=0.172 e/s/pix)
  - t=600s → m_lim=21.11  (n_pix=41, B=0.172 e/s/pix)
  - t=900s → m_lim=21.37  (n_pix=41, B=0.172 e/s/pix)
  - t=450s → m_lim=20.91  (n_pix=41, B=0.172 e/s/pix)
  - t=540s → m_lim=21.04  (n_pix=41, B=0.172 e/s/pix)
  - t=570s → m_lim=21.07  (n_pix=41, B=0.172 e/s/pix)
  - t=580s → m_lim=21.08  (n_pix=41, B=0.172 e/s/pix)
  - t=555s → m_lim=21.06  (n_pix=41, B=0.172 e/s/pix)
  - t=54

**What dependency injection gives you:**

- Tools share resources (database connections, configuration) without using global variables.
- The agent behaviour changes based on the concrete `deps` instance passed at run time — useful for multi-user systems, unit testing with mock data, or switching between telescope configurations.
- All of this is **type-safe**: if you annotate `ctx: RunContext[ObservatoryDeps]` incorrectly, a static type checker will warn you.

---
## Section 4 — Multi-Agent Workflows

Complex tasks often benefit from **specialisation**: one agent handles one sub-problem, another handles a different one. PydanticAI supports this natively — one agent can call another agent inside a tool.

The pattern:
- A **coordinator agent** receives the high-level task.
- It has tools that internally call **specialist agents** and return their outputs.
- Usage (token counts, API calls) can be propagated across agents via `ctx.usage`.

### Example: Photometric Pipeline with Specialist Agents

In the example below we will give a coordinator agent some high level information about an astrophysical object that we want to observe and the instrument we have available. We expect a detailed recommendation about how to observe this object based on what the most likely class of the object (given by a classifier agent) and best observational practices (given by a photometry agent).

We build a small pipeline with three agents: a classifier, a photometry advisor, and a coordinator that orchestrates them. The diagram below shows how a single user prompt flows through the system:

```
                       user prompt
                            │
                            ▼
              ┌──────────────────────────────┐
              │     coordinator_agent        │
              │  output_type=PipelineReport  │
              └──┬────────────────────────┬──┘
                 │ ① tool call            │ ② tool call
                 │   classify_source(…)   │   get_photometry_strategy(…)
                 ▼                        ▼
       ┌────────────────────┐  ┌──────────────────────────┐
       │ classifier_agent   │  │   photometry_agent       │
       │   → SourceType     │  │ → PhotometryStrategy     │
       └─────────┬──────────┘  └────────────┬─────────────┘
                 │                          │
                 │   typed results          │
                 └────────────┬─────────────┘
                              │ returned to coordinator
                              ▼
              ┌──────────────────────────────┐
              │  coordinator synthesises a   │
              │  final PipelineReport        │
              │  (validated by Pydantic)     │
              └──────────────┬───────────────┘
                             ▼
                       pipeline_result.output
                       (typed Python object)
```

**Reading the code below, keep this in mind:**

- Each agent has its own `output_type` — the LLM sees only the schema relevant to its sub-task.
- The coordinator's tools are **not** Python computations: they are *thin wrappers* around `await specialist_agent.run(...)`. The LLM driving the coordinator decides *when* and *with what arguments* to call them.
- `usage=ctx.usage` is passed into each sub-agent call so token counts bubble up to the parent — `pipeline_result.usage()` at the end is the **total** across all three agents.

In [28]:
from pydantic import BaseModel, Field
from pydantic_ai import Agent, RunContext
from typing import Literal


# ============================================================
# Specialist Agent 1: Source Classifier
# ============================================================

class SourceType(BaseModel):
    source_class: Literal["star", "galaxy", "nebula", "quasar", "transient", "unknown"]
    confidence: float = Field(ge=0.0, le=1.0)
    reasoning: str


classifier_agent = Agent(
    "anthropic:claude-sonnet-4-6",
    output_type=SourceType,
    instructions="You are a source classification expert. Be concise.",
)


# ============================================================
# Specialist Agent 2: Photometry Strategy Advisor
# ============================================================

class PhotometryStrategy(BaseModel):
    method: str
    required_calibrations: list[str]
    estimated_precision_mag: float
    caveats: str


photometry_agent = Agent(
    "anthropic:claude-sonnet-4-6",
    output_type=PhotometryStrategy,
    instructions=(
        "You are a photometric data reduction expert. Be concise. "
        "Given a source type and instrument, recommend a photometry strategy."
    ),
)


# ============================================================
# Coordinator Agent: orchestrates the specialists
# ============================================================

class PipelineReport(BaseModel):
    classification: SourceType
    photometry_strategy: PhotometryStrategy
    executive_summary: str


coordinator_agent = Agent(
    "anthropic:claude-sonnet-4-6",
    output_type=PipelineReport,
    instructions=(
        "Pipeline coordinator. Use the tools to classify the source and "
        "design a photometry strategy, then produce a concise report."
    ),
)


@coordinator_agent.tool
async def classify_source(ctx: RunContext[None], description: str) -> dict:
    """Classify an astronomical source from a description."""
    # Propagate token usage upward via ctx.usage
    result = await classifier_agent.run(description, usage=ctx.usage)
    return result.output.model_dump()


@coordinator_agent.tool
async def get_photometry_strategy(ctx: RunContext[None], source_type: str, instrument: str) -> dict:
    """Recommend a photometry strategy for the given source type and instrument."""
    prompt = f"Source type: {source_type}. Instrument: {instrument}."
    result = await photometry_agent.run(prompt, usage=ctx.usage)
    return result.output.model_dump()


# Run the full pipeline
pipeline_result = await coordinator_agent.run(
    "Unresolved blue point source, variable on day timescales. "
    "Instrument: 1-m telescope with a CCD camera in BVRI filters."
)

report = pipeline_result.output
print("=== PIPELINE REPORT ===")
print(f"Class       : {report.classification.source_class} ({report.classification.confidence:.0%})")
print(f"Reasoning   : {report.classification.reasoning}")
print()
print(f"Method      : {report.photometry_strategy.method}")
print(f"Calibrations: {', '.join(report.photometry_strategy.required_calibrations)}")
print(f"Precision   : ±{report.photometry_strategy.estimated_precision_mag:.3f} mag")
print()
print(f"Summary     : {report.executive_summary}")
print()
print(f"Usage: {pipeline_result.usage()}")

=== PIPELINE REPORT ===
Class       : quasar (72%)
Reasoning   : An unresolved blue point source that is variable on day timescales is strongly suggestive of a quasar (AGN). Quasars appear as unresolved point sources indistinguishable from stars at typical survey resolution, have characteristically blue colors due to their accretion disk emission, and exhibit stochastic variability on timescales of days to years. While a blue variable star (e.g., a cataclysmic variable or Be star) or a transient (e.g., a blazar flare) could also match some criteria, the combination of blue color, point-source morphology, and day-timescale variability most consistently points to a quasar/AGN.

Method      : Aperture photometry with differential/absolute calibration via standard star fields. Use a circular aperture scaled to ~2–3× the PSF FWHM centered on the quasar nucleus. Apply sky background subtraction using an annular region well clear of the source. Observe Landolt or similar standard star fields 

In [30]:
# Run the full pipeline
pipeline_result = await coordinator_agent.run(
    "Rapidly rising source next to a nearby galaxy. "
    "Instrument: 1-m telescope with a CCD camera in BVRI filters."
)

report = pipeline_result.output
print("=== PIPELINE REPORT ===")
print(f"Class       : {report.classification.source_class} ({report.classification.confidence:.0%})")
print(f"Reasoning   : {report.classification.reasoning}")
print()
print(f"Method      : {report.photometry_strategy.method}")
print(f"Calibrations: {', '.join(report.photometry_strategy.required_calibrations)}")
print(f"Precision   : ±{report.photometry_strategy.estimated_precision_mag:.3f} mag")
print()
print(f"Summary     : {report.executive_summary}")
print()
print(f"Usage: {pipeline_result.usage()}")

=== PIPELINE REPORT ===
Class       : transient (85%)
Reasoning   : A rapidly rising source located near a nearby galaxy strongly suggests a transient event. This description is characteristic of phenomena such as a supernova, nova, tidal disruption event (TDE), or gamma-ray burst afterglow associated with the host galaxy. The key indicators are: (1) 'rapidly rising' implies a sudden brightening, typical of explosive or eruptive transient events, and (2) proximity to a nearby galaxy suggests an extragalactic transient rather than a foreground stellar or Galactic source.

Method      : Aperture photometry with differential/ensemble photometry against local field standards. For each BVRI band: (1) identify non-variable field stars as local comparisons; (2) perform fixed-aperture photometry on the transient and comparison stars using an aperture radius ~1.5–2× the FWHM of the PSF; (3) apply differential photometry relative to the ensemble to derive instrumental magnitudes; (4) transform t

**Notes on multi-agent patterns:**

- Specialist agents are called with `await agent.run(...)` inside `@agent.tool` functions.
- Passing `usage=ctx.usage` propagates token accounting upward, so you can track total cost across the whole pipeline.
- Each specialist has its own `output_type`; outputs are validated independently.
- You can mix providers across agents (fast model for classification, more capable one for the final report).

---
## Summary

| Concept | What it does | When to use it |
|---|---|---|
| `Agent(output_type=MyModel)` | Forces the LLM to return validated structured data | Whenever you need reliable, typed output |
| `@agent.tool_plain` | Registers a stateless Python function the LLM can call | Computations and lookups without shared state |
| `@agent.tool` + `RunContext` | Registers a function with access to runtime dependencies | DB connections, API keys, session state |
| `deps_type` + `@agent.instructions` | Injects context into the system prompt dynamically | Personalisation, multi-user applications |
| Agent calling Agent | One agent delegates sub-tasks to specialist agents | Complex multi-step pipelines |


### References

- [PydanticAI docs](https://ai.pydantic.dev/) · [Pydantic v2 docs](https://docs.pydantic.dev/) · [GitHub](https://github.com/pydantic/pydantic-ai)